# Python basics

**Objective:** Combine the Python features used most often in maintainable backend services.

## Type hints

In [ ]:
def total(prices: list[float]) -> float:
    return sum(prices)


print(total([9.99, 4.50]))

## Exceptions

In [ ]:
class InsufficientBalanceError(ValueError):
    pass


def withdraw(balance: float, amount: float) -> float:
    if amount > balance:
        raise InsufficientBalanceError("insufficient balance")
    return balance - amount


try:
    withdraw(50, 80)
except InsufficientBalanceError as error:
    print("Handled:", error)

## Data classes

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class User:
    id: int
    email: str


print(User(id=1, email="ada@example.com"))

## Project structure and dependencies

In [ ]:
layers = {
    "routers/": "HTTP endpoints",
    "services/": "business rules",
    "models.py": "data models",
    "tests/": "automated verification",
}
dependencies = {
    "runtime": ["fastapi"],
    "development": ["ruff"],
}

print(layers)
print(dependencies)

## Polished version

A service slice combines typed data, an interface, an adapter, a domain error, and business logic.

In [ ]:
from dataclasses import dataclass
from typing import Optional, Protocol


@dataclass(frozen=True)
class Customer:
    id: int
    email: str


class CustomerRepository(Protocol):
    def find_by_email(self, email: str) -> Optional[Customer]: ...
    def add(self, email: str) -> Customer: ...


class DuplicateEmailError(ValueError):
    pass


class MemoryCustomerRepository:
    def __init__(self) -> None:
        self.customers: list[Customer] = []

    def find_by_email(self, email: str) -> Optional[Customer]:
        return next(
            (customer for customer in self.customers if customer.email == email),
            None,
        )

    def add(self, email: str) -> Customer:
        customer = Customer(id=len(self.customers) + 1, email=email)
        self.customers.append(customer)
        return customer


class CustomerService:
    def __init__(self, customers: CustomerRepository) -> None:
        self.customers = customers

    def register(self, email: str) -> Customer:
        normalized = email.strip().lower()
        if self.customers.find_by_email(normalized):
            raise DuplicateEmailError(normalized)
        return self.customers.add(normalized)


service = CustomerService(MemoryCustomerRepository())
print(service.register("Ada@Example.com"))